# Hierarchical Clustering - UPGMA

---
## Learning Objectives

1. Implement a distance-based phylogeny neighbor joining algorithm
1. Understanding of hierarchical clustering
1. Manipulation of arrays and lists in Python


---
## Imports
You will need to install a two new packages to properly render your phylogenetic trees. Using your preferred package manager, install both [`biopython`](https://biopython.org/) and `matplotlib` into your virtual environments.

For example:
```bash
$ micromamba create -n module06 -c bioconda -c conda-forge python numpy matplotlib biopython
```

In [41]:
%pip install biopython


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [42]:
from typing import List, Tuple, Dict
from io import StringIO

import matplotlib.pyplot as plt
from Bio import Phylo
import numpy as np
import math

---
## Background

Today we will implement a distance-based phylogenetic tree construction method using the Neighbor-Joining (NJ) algorithm and Smith-Waterman local alignment scores. Unlike UPGMA (Unweighted Pair Group Method using Arithmetic averages), NJ produces unrooted trees and does not assume a constant evolutionary rate across lineages, making it more biologically realistic for analyzing sequence relationships.

We will use HIV-1 reverse transcriptase sequences to construct our phylogenetic tree. The Smith-Waterman algorithm will be used to generate pairwise local alignment scores, which will then be converted into distances for tree construction. This approach is particularly suitable for HIV sequence analysis as it:
1. Handles sequence variations effectively through local alignment
2. Accounts for potential rate heterogeneity across different viral strains
3. Does not assume a molecular clock

The ultimate output will be an unrooted phylogenetic tree representing the evolutionary relationships between the HIV-1 sequences, visualized using the ete3 library. This method provides insights into viral diversity and evolutionary patterns while avoiding the assumptions of simpler hierarchical clustering approaches.

The key innovations of this implementation are:
- Use of Smith-Waterman for sensitive local alignment scoring
- Implementation of Neighbor-Joining for unrooted tree construction
- More biologically realistic evolutionary model

## Neighbor-Joining Algorithm

**Input**: Distance matrix $D$ for $n$ sequences

**Initialization**:
- Let $n$ be the number of sequences
- Assign each sequence $i$ to its own leaf node
- Initialize tree $T$ with leaf nodes

**Iteration**:
While $n > 2$:
1. Calculate Q-matrix where:
   $Q(i,j) = (n-2)d(i,j) - \sum_{k=1}^n d(i,k) - \sum_{k=1}^n d(j,k)$

2. Find pair $(i,j)$ with minimum $Q(i,j)$

3. Calculate branch lengths:
   $d_i = \frac{d(i,j) + (r_i - r_j)/(n-2)}{2}$
   $d_j = d(i,j) - d_i$
   where $r_i = \sum_{k=1}^n d(i,k)$

4. Create new node $k$
   - Add branches from $k$ to $i$ and $j$ with lengths $d_i$ and $d_j$

5. Update distances to remaining nodes $x$:
   $d(k,x) = \frac{d(i,x) + d(j,x) - d(i,j)}{2}$

6. Remove nodes $i$ and $j$
7. Add node $k$ to active nodes
8. $n = n - 1$

**Termination**:
When $n = 2$ with remaining nodes $i$ and $j$:
- Add final branch between $i$ and $j$ with length $d(i,j)$
- Return unrooted tree $T$

---
## Distance metrics for comparing sequences

### Smith-Waterman
This was previously implemented. Feel free to import if you know how...or Copy & Paste.



In [43]:
def read_fasta(filename: str) -> Dict[str, str]:
    """Reads sequences from FASTA file.

    Args:
        filename (str): Path to FASTA file containing HIV RT sequences

    Returns:
        Dict[str, str]: Dictionary mapping sequence IDs to sequences

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> len(seqs) > 0
        True
    """
    
    sequences: Dict[str, str] = {}
    current_id = None
    current_sequence = []

    with open(filename, "r") as file:
        for line in file:
            line = line.strip()

            if line == "":
                continue

            if line.startswith(">"):
                if current_id is not None:
                    sequences[current_id] = "".join(current_sequence)

                header = line[1:].strip()
                current_id = header.split()[0] if header else ""
                current_sequence = []
            else:
                current_sequence.append(line)

    if current_id is not None:
        sequences[current_id] = "".join(current_sequence)

    return sequences

In [44]:

# Test read_fasta()
sequences = read_fasta("data/lafayette_SARS_RT.fasta")

print("Number of sequences:", len(sequences))

# Display the first 3 sequences
for seq_id, sequence in list(sequences.items())[:3]:
    print(f"\nSequence ID: {seq_id}")
    print(f"Length: {len(sequence)}")
    print(f"First 50 characters: {sequence[:50]}")


Number of sequences: 20

Sequence ID: gi|24209940|gb|AAN41462.1|
Length: 229
First 50 characters: PISPIETVPVKLKPGMDGPKVKQWPLTEEKIKALVEICTEMEKEGKISKI

Sequence ID: gi|24209942|gb|AAN41463.1|
Length: 229
First 50 characters: PISPIETVPVKLKPGMDGPKVKQWPLTEEKIKALVEICTEMEKEGKISKI

Sequence ID: gi|24209944|gb|AAN41464.1|
Length: 229
First 50 characters: PISPIETVPVKLKPGMDGPKVKQWPLTEEKIKALVEICTEMEKEGKISKI


In [45]:
def traceback_score(
    sequence1: str,
    sequence2: str,
    traceback_matrix: np.ndarray,
    maximum_position: Tuple[int, int],
    match_score: float = 2.0,
    mismatch_penalty: float = -1.0,
    gap_penalty: float = -1.0
) -> Tuple[str, str, float]:
    """Reconstructs the optimal Smith-Waterman local alignment
    and calculates its alignment score.

    Args:
        sequence1 (str): First HIV RT sequence
        sequence2 (str): Second HIV RT sequence
        traceback_matrix (np.ndarray): Matrix containing traceback
            directions (0=END, 1=DIAG, 2=UP, 3=LEFT)
        maximum_position (Tuple[int, int]): Position of the highest
            score in the Smith-Waterman scoring matrix
        match_score (float): Score for matching characters
        mismatch_penalty (float): Penalty for mismatched characters
        gap_penalty (float): Penalty for gaps

    Returns:
        Tuple[str, str, float]: Aligned sequence1, aligned sequence2,
            and the raw local alignment score

    Examples:
        >>> seq1 = "ACGT"
        >>> seq2 = "ACGA"
        >>> traceback = np.array([
        ...     [0, 0, 0, 0, 0],
        ...     [0, 1, 3, 3, 3],
        ...     [0, 2, 1, 3, 3],
        ...     [0, 2, 2, 1, 3],
        ...     [0, 2, 2, 2, 1]
        ... ])
        >>> traceback_score(seq1, seq2, traceback, (3, 3))
        ('ACG', 'ACG', 6.0)
    """
    aligned_seq1 = []
    aligned_seq2 = []
    alignment_score = 0.0

    current_row, current_col = maximum_position

    while current_row > 0 and current_col > 0:
        current_move = traceback_matrix[current_row, current_col]
        
        if current_move == 0:
            break

        #check match and mismatch
        elif current_move == 1:

            char1 = sequence1[current_row - 1]
            char2 = sequence2[current_col - 1]

            aligned_seq1.append(char1)
            aligned_seq2.append(char2)

            if char1 == char2:
                alignment_score += match_score
            else:
                alignment_score += mismatch_penalty

            current_row -= 1
            current_col -= 1

        # UP: Gap in sequence 2
        elif current_move == 2:
            aligned_seq1.append(sequence1[current_row - 1])
            aligned_seq2.append("-")

            alignment_score += gap_penalty
            current_row -= 1

        # LEFT: Gap in sequence 1
        elif current_move == 3:
            aligned_seq1.append("-")
            aligned_seq2.append(sequence2[current_col - 1])

            alignment_score += gap_penalty
            current_col -= 1

    # Reverse lists and join into strings
    aligned_seq1 = "".join(reversed(aligned_seq1))
    aligned_seq2 = "".join(reversed(aligned_seq2))

    return  aligned_seq1, aligned_seq2, alignment_score
    


In [46]:
def smith_waterman(
    sequence1: str,
    sequence2: str,
    match_score: float = 2.0,
    mismatch_penalty: float = -1.0,
    gap_penalty: float = -1.0
) -> float:
    """Implements Smith-Waterman local alignment algorithm.

    Args:
        sequence1 (str): First HIV RT sequence
        sequence2 (str): Second HIV RT sequence
        match_score (float): Score for matching characters
        mismatch_penalty (float): Penalty for mismatched characters
        gap_penalty (float): Penalty for gaps

    Returns:
        float: Normalized similarity score between sequences

    Examples:
        >>> seq1 = "ACGT"
        >>> seq2 = "ACGT"
        >>> smith_waterman(seq1, seq2)
        1.0
    """
    n = len(sequence1)
    m = len(sequence2)

    # Check if there is empty sequence
    if n == 0 or m == 0:
        return 0.0
    
    # Initialize matrizes
    score_matrix = np.zeros((n + 1, m + 1), dtype = float)
    traceback_matrix = np.zeros((n + 1, m + 1), dtype = int)

    max_score = 0.0
    maximum_position = None

    # Foward sweep
    for i in range(1, n + 1):
        for j in range(1, m + 1):

            if sequence1[i - 1] == sequence2[j - 1]:
                substitution = match_score
            else:
                substitution = mismatch_penalty

            diagonal = score_matrix[i - 1, j - 1] + substitution
            up = score_matrix[i - 1, j] + gap_penalty
            left = score_matrix[i, j - 1] + gap_penalty

            score = max(0.0, diagonal, up, left)

            # Score Matrix and traceback direction
            score_matrix[i, j] = score

            if score == 0: # END
                traceback_matrix[i, j] = 0 
            
            elif score == diagonal: # DIAGONAL
                traceback_matrix[i, j] = 1 

            elif score == up: # UP 
                traceback_matrix[i, j] = 2 

            else:  # LEFT
                traceback_matrix[i, j] = 3 
            
            if score > max_score:
                max_score = score
                maximum_position = (i, j)

    # Handle no positive alignment
    if max_score == 0:
        return 0.0
    
    # Backward Sweep
    aligned_seq1, aligned_seq2, alignment_score = traceback_score(
        sequence1,
        sequence2,
        traceback_matrix,
        maximum_position,
        match_score,
        mismatch_penalty,
        gap_penalty
    )

    # Calculate Self_alignment scores
    score_uu = n * match_score
    score_vv = m * match_score

    normalized_score = alignment_score / math.sqrt(score_uu * score_vv)

    return round(float(normalized_score), 2)


In [47]:
# Test 1: Identical sequences
print(smith_waterman("ACGT", "ACGT"))
# Expected: 1.0

# Test 2: Partial similarity
print(smith_waterman("ACGT", "ACGA"))
# Expected: 0.75

# Test 3: No matching characters
print(smith_waterman("AAAA", "TTTT"))
# Expected: 0.0

# Test 4: Empty sequence
print(smith_waterman("", "ACGT"))
# Expected: 0.0

# Test 5: Different lengths
print(smith_waterman("ACGT", "TTACGTAA"))
# Expected: approximately 0.71

1.0
0.75
0.0
0.0
0.71


In [48]:
def build_distance_matrix(sequences: Dict[str, str]) -> Tuple[np.ndarray, List[str]]:
    """Builds DISTANCE (not similiarity) matrix using Smith-Waterman scores.

    Args:
        sequences (Dict[str, str]): Dictionary of HIV RT sequences

    Returns:
        Tuple[np.ndarray, List[str]]: Distance matrix and list of sequence IDs

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> mat[0:3, 0:3]  # Show 3x3 slice of distance matrix
        array([[0.000, 0.004, 0.012],
              [0.004, 0.000, 0.012],
              [0.012, 0.012, 0.000]])
        >>> len(ids)  # Number of sequences
        19
    """
    ids = list(sequences.keys())
    n = len(ids)

    # Build n*n matrix, initialized with 0.0
    distance_matrix = np.zeros((n, n), dtype=float)

    # Calculate pairwise distances
    for i in range(0, n - 1):

        # Get first sequence
        seq1 = sequences[ids[i]]

        for j in range(i + 1, n):

            # Get second sequence
            seq2 = sequences[ids[j]]

            # Smith-Waterman comparison
            similarity = smith_waterman(seq1, seq2)

            # Convert similarity into distance
            distance = 1.0 - similarity

            # Update symmetric distance matrix
            distance_matrix[i][j] = distance
            distance_matrix[j][i] = distance

    return distance_matrix, ids

In [49]:
sequences = read_fasta("data/lafayette_SARS_RT.fasta")

distance_matrix, ids = build_distance_matrix(sequences)

print("Matrix shape:", distance_matrix.shape)
print("Sequence count:", len(ids))

print("\nFirst three rows and columns:")
print(np.round(distance_matrix[:3, :3], 4))

Matrix shape: (20, 20)
Sequence count: 20

First three rows and columns:
[[0.   0.01 0.05]
 [0.01 0.   0.04]
 [0.05 0.04 0.  ]]


In [50]:
def neighbor_joining(
    distance_matrix: np.ndarray,
    labels: List[str]
) -> str:
    """Implements Neighbor-Joining algorithm for phylogenetic tree construction.

    Args:
        distance_matrix (np.ndarray): Distance matrix from Smith-Waterman scores
        labels (List[str]): Sequence identifiers

    Returns:
        str: Newick format tree string

    Examples:
        >>> seqs = read_fasta("lafayette_SARS_RT.fasta")
        >>> mat, ids = build_distance_matrix(seqs)
        >>> tree = neighbor_joining(mat, ids)
        >>> tree.startswith("((DM1:0.002,DM2:0.002)")  # Start of Newick string
        True
        >>> tree.count(",")  # Number of separators in tree
        18
    """
    # Copy so the original matrix is no modified 
    current_distances = np.array(distance_matrix, dtype = float, copy = True)
    active_nodes = list(labels)
    number_of_nodes = len(active_nodes)

    newick_strings = {}

    for sequence_id in active_nodes:
        newick_strings[sequence_id] = sequence_id
    
    internal_node_counter = 0

    # continue while more than 2 nodes remain
    while number_of_nodes > 2:

        # Step 1 - Calculate row sums and Q-matrix
        row_sums = []

        for row in range(number_of_nodes):
            total_distance = 0.0 
            
            for col in range(number_of_nodes):
                total_distance += current_distances[row, col]

            row_sums.append(total_distance)
        
        # Building Q matrix
        neighbor_joining_matrix = np.full((number_of_nodes, number_of_nodes), np.inf)

        # Step 2 - Calculate Q values and find minimum pair
        minimum_joining_value = float("inf")
        first_node_index = None
        second_node_index = None

        for row in range(number_of_nodes):
            for col in range(row + 1, number_of_nodes):

                joining_value = ((number_of_nodes-2) * current_distances[row, col] 
                                 - row_sums[row] - row_sums[col])

                neighbor_joining_matrix[row, col] = joining_value
                neighbor_joining_matrix[col, row] = joining_value

                if joining_value < minimum_joining_value:
                    minimum_joining_value = joining_value
                    first_node_index = row
                    second_node_index = col

        # Step 3 - Calcula Branch Lengths
        selected_distance = current_distances[first_node_index, second_node_index]

        first_branch_length = 0.5 * (selected_distance + (row_sums[first_node_index] - row_sums[second_node_index])/ (number_of_nodes - 2))

        second_branch_length = selected_distance - first_branch_length

        # Step 4  - Create new internal node 
        first_node = active_nodes[first_node_index]
        second_node = active_nodes[second_node_index]

        new_node = f"INTERNAL_{internal_node_counter}"
        internal_node_counter += 1

        while new_node in labels:
            new_node = f"INTERNAL_{internal_node_counter}"
            internal_node_counter += 1
        
        newick_strings[new_node] = (
            f"({newick_strings[first_node]}:{first_branch_length:.6f},"
            f"({newick_strings[second_node]}:{second_branch_length:.6f})"
        )

        # Step 5 - Identify remaining nodes
        remaining_indices = []

        for index in range(number_of_nodes):
            if index != first_node_index and index != second_node_index:
                remaining_indices.append(index)
        
        remaining_nodes = []
        for index in remaining_indices:
            remaining_nodes.append(active_nodes[index])

        # New Updated distance Matrix
        new_size = number_of_nodes - 1

        updated_distances = np.zeros((new_size, new_size), dtype = float)

        for new_row in range(len(remaining_indices)):
            for new_col in range(len(remaining_indices)):
                original_row = remaining_indices[new_row]
                original_col = remaining_indices[new_col]

                updated_distances[new_row, new_col] = (current_distances[original_row, original_col])
        
        # New Distances
        for new_index in range(len(remaining_indices)):
            original_index = remaining_indices[new_index]

            new_distance = 0.5 * (
                current_distances[first_node_index, original_index] 
                + current_distances[second_node_index, original_index]
                - selected_distance
            )
            updated_distances[new_index, new_size - 1] = new_distance
            updated_distances[new_size - 1, new_index] = new_distance

        # Step 6 - Remove joined nodes
        del newick_strings[first_node]
        del newick_strings[second_node]

        # Step 7 - Add the new internal node
        remaining_nodes.append(new_node)
        active_nodes = remaining_nodes
        current_distances = updated_distances

        # Step 8 - Decrease number of active nodes
        number_of_nodes -= 1
    
    # Termination 
    first_remaining_node = active_nodes[0]
    second_remaining_node = active_nodes[1]

    final_distance = current_distances[0, 1]
    final_branch_length = final_distance / 2.0

    final_tree = (
        f"({newick_strings[first_remaining_node]}):"
        f"({final_branch_length:.6f}),"
        f"({newick_strings[second_remaining_node]}):"
        f"({final_branch_length:.6f});"
    )

    return final_tree


In [51]:

test_distance_matrix = np.array([
    [0, 2, 6, 6],
    [2, 0, 6, 6],
    [6, 6, 0, 2],
    [6, 6, 2, 0]
], dtype=float)

test_sequence_ids = ["A", "B", "C", "D"]

test_tree = neighbor_joining(
    test_distance_matrix,
    test_sequence_ids
)

print(test_tree)

assert test_tree.endswith(";")
assert test_tree.count(",") == 3

print("Neighbor-Joining test passed!")


((A:1.000000,(B:1.000000)):(2.000000),((C:1.000000,(D:1.000000)):(2.000000);
Neighbor-Joining test passed!


In [52]:
def plot_tree(newick_tree: str) -> None:
    """Plots a phylogenetic tree from a Newick string using Biopython and Matplotlib.

    The tree is rendered in a rectangular layout with smaller leaf labels and
    added margins to reduce overlap between labels and branches.

    Args:
        newick_tree (str): Tree in Newick format

    Returns:
        None: Displays the plotted tree
    """
    handle = StringIO(newick_tree)
    tree = Phylo.read(handle, "newick")

    fig, ax = plt.subplots(figsize=(8, 10))

    Phylo.draw(
        tree,
        axes=ax,
        do_show=False,
        label_func=lambda clade: clade.name if clade.is_terminal() else None,
    )
    for text in ax.texts:
        text.set_fontsize(6)              # shrink labels
    ax.margins(x=0.1, y=0.05)             # extra padding around tree

    plt.tight_layout()
    plt.show()


In [53]:
if __name__ == "__main__":
    # Read HIV RT sequences
    sequences = read_fasta("data/lafayette_SARS_RT.fasta")
    
    # Build distance matrix using Smith-Waterman
    dist_matrix, seq_ids = build_distance_matrix(sequences)
    
    # Generate unrooted tree using Neighbor-Joining
    tree = neighbor_joining(dist_matrix, seq_ids)
    
    # Plot tree
    plot_tree(tree)

NewickError: Mismatch, 40 open vs 22 close parentheses.